<h1> Information Retrieval using ChatGPT and the OpenAI Python API library

In [7]:
import openai
from openai import OpenAI
import pandas as pd

In [8]:
# Prepare the connection through the OpenAI Python API library. Credentials will be read from the environment file by default.
client = OpenAI()

In [9]:
FILE_PATH = f"../data/spotify_reviews_post-2023.json"

# Fetch relevant data
try:
    df = pd.read_csv('../data/spotify_reviews.csv')
except Exception as e:
    raise Exception(f"Failed to read CSV file: {e}")

# Filter and format relevant data
df = df.drop(columns=['reviewId', 'userName', 'score', 'thumbsUpCount', 'reviewCreatedVersion'])
df['at'] = pd.to_datetime(df['at'])
df = df[df['at'] > '2023-01-01']
 
# Save data as JSON file
try:
    df.to_json(FILE_PATH, orient='records')
except Exception as e:
    raise Exception(f"Failed to write JSON file: {e}")

In [10]:
system_prompt = """
                You are a helpful assistant analyzing Spotify app reviews. 
                Be precise and focus on specific features which are highlighted in the reviews. 
                Limit your response to the data available. 
                Do not provide general information.
                Provide your response in a suitable markdown format. 
                """

In [12]:
# Create the Assistant
print("Creating assistant...")
try:
    assistant = client.beta.assistants.create(
        name="Spotify Review Analyzer",
        instructions=system_prompt,
        model="gpt-3.5-turbo",
        tools=[{"type": "file_search"}],
        temperature=0
    )
except Exception as e:
    raise Exception(f"Failed to create assistant: {e}")

# Create a vector store to store the data
print("Creating vector store...")
try:
    vector_store = client.beta.vector_stores.create(name=f"Spotify Review Vector Store")
except Exception as e:
    raise Exception(f"Failed to create vector store: {e}")

# Prepare files for upload to OpenAI
file_paths = [FILE_PATH]
file_streams = [open(path, "rb") for path in file_paths]

# Use SDK helper to upload the files, add them to the vector store and poll the status of the file batch for completion.
print("Uploading files...")
try:
    file_batch = client.beta.vector_stores.file_batches.upload_and_poll(
    vector_store_id=vector_store.id, files=file_streams
)
except Exception as e:
    raise Exception(f"Failed to upload files: {e}")
    
# You can print the status and the file counts of the batch to see the result of this operation.
print("Finished uploading files.")
print(file_batch.status)
print(file_batch.file_counts)

# Update the assistant with the vector store
print("Updating assistant with vector store...")
try:
    assistant = client.beta.assistants.update(
        assistant_id=assistant.id,
        tool_resources={"file_search": {"vector_store_ids": [vector_store.id]}},
    )
except Exception as e:
    raise Exception(f"Failed to update assistant: {e}")

Creating assistant...
Creating vector store...
Uploading files...
Finished uploading files.
completed
FileCounts(cancelled=0, completed=1, failed=0, in_progress=0, total=1)
Updating assistant with vector store...


In [18]:
# Create the Thread
try:
    thread = openai.beta.threads.create()
except Exception as e:
    raise Exception(f"Failed to create thread: {e}")

# Add individual review to the thread
def add_user_message(message):
    try:
        openai.beta.threads.messages.create(
            thread_id=thread.id,
            role="user",
            content=message
        )
    except Exception as e:
        raise Exception(f"Failed to add user message: {e}")

# Function to run the assistant on the thread
def run_assistant():
    try:
        run = openai.beta.threads.runs.create(
            thread_id=thread.id,
            assistant_id=assistant.id
        )
        return run
    except Exception as e:
        raise Exception(f"Failed to run assistant: {e}")

# Function to retrieve and process assistant's response
def get_assistant_response(run_id):
    try:
        messages = openai.beta.threads.messages.list(
            thread_id=thread.id
        )
        response_message = messages.data[0]
        return response_message.content[0].text.value
    except Exception as e:
        raise Exception(f"Failed to get assistant response: {e}")

# Function to add user message and run the assistant
def add_user_message_and_run(user_prompt):
    add_user_message(user_prompt)
    run = run_assistant()
    while run.status != "completed":
        run = openai.beta.threads.runs.retrieve(thread_id=thread.id, run_id=run.id)
    assistant_response = get_assistant_response(run.id)
    return assistant_response

In [14]:
prompt_likes = """
                Please extract the top 10 specific features that users like the most about the Spotify app based on the reviews data.
                """

print(add_user_message_and_run(prompt_likes))

Based on the reviews data, the top 10 specific features that users like the most about the Spotify app are:

1. Excellent sound quality and a wide selection of genres for all ages and tastes.
2. Easy navigation and reasonable pricing.
3. Personalized playlists that intuitively cater to users' music preferences.
4. Ability to create playlists as needed.
5. Continuous playback even when using other apps.
6. Extensive music and podcast selection.
7. User-friendly interface with an interactive music listening experience.
8. Ability to keep songs playing based on user preferences.
9. Large database of music outside of YouTube and YouTube Music.
10. Seamless and enjoyable listening experience with a smooth user interface and helpful recommendation algorithms.

These features were highlighted positively by users in the reviews of the Spotify app【4:0†source】【4:1†source】【4:2†source】【4:3†source】.


In [15]:
prompt_dislikes = """
                    Please extract the top 10 specific features that users dislike the most about the Spotify app based on the reviews data.
                """

print(add_user_message_and_run(prompt_dislikes))

Based on the reviews data, the top 10 specific features that users dislike the most about the Spotify app are:

1. Slow performance, glitches, crashes, and music playback issues, even for premium subscribers【8:0†source】.
2. Pricing perceived as overpriced and glitches in music playback even with premium subscription【8:0†source】.
3. Issues with the like button placement and repetitive song suggestions in playlists【8:0†source】.
4. Slow loading, shuffling problems, and frequent offline mode errors【8:0†source】.
5. Annoyances with the multiple device feature and difficulties in playing songs【8:0†source】.
6. Visual clutter and frustration with the app interface【8:1†source】.
7. Limitations on skips, ads, and lyrics for non-premium users【8:1†source】.
8. Changes in the app's appearance and difficulty in finding desired music【8:1†source】.
9. Random song plays, shuffle issues, and hidden settings button【8:1†source】.
10. Inconvenient AI features, removal of liked songs functionality, and limitatio

In [16]:
prompt_wants = """
                Please extract the top 10 specific features that users want to see in the Spotify app based on the reviews data.
                """

print(add_user_message_and_run(prompt_wants))

Based on the reviews data, the top 10 specific features that users want to see in the Spotify app are:

1. Ability to add multiple Spotify accounts and switch between them seamlessly without logging out.
2. Feature request for playlist folders to create new folders and add playlists to existing ones on the Android app.
3. Request for the return of the "hearts for songs" feature for easier song liking.
4. Desire for a "Shazam" type option to identify songs heard and add them to playlists immediately.
5. Request for a speed changer feature in the app.
6. Improvement in the lyrics display functionality, especially for songs that users are unable to load lyrics for.
7. Desire for a lock screen widget to be reinstated for easier control and navigation.
8. Request for a "Shuffle" feature that does not repeat songs by the same artists excessively.
9. Implementation of a feature to listen to specific parts of songs and the ability to repeat specific sections.
10. Improvement in the app's speed

In [17]:
prompt_bugs = """
                Please extract the top 10 specific bugs that users have reported in the Spotify app based on the reviews data.
                """

print(add_user_message_and_run(prompt_bugs))

Based on the reviews data, the top 10 specific bugs that users have reported in the Spotify app are:

1. Shuffle being auto-applied despite being turned off, auto-play when opening the app due to Bluetooth headphones being on, and issues with liked songs being duplicated or not appearing in the list【16:0†source】.
2. Annoyance with the smart shuffle feature that plays random songs or interrupts playlists, with users unable to turn it off or resume music playback【16:0†source】.
3. Inability to use the app in the background without music randomly stopping and issues with skipping like a scratched record when using Spotify in the car【16:0†source】.
4. Music freezing mid-playback even with downloaded songs and songs not transitioning to the next track, leading to freezing at the end of songs【16:0†source】.
5. Difficulty in seamlessly transitioning to another playlist after finishing one, especially when offline, requiring manual intervention to play the next album or playlist【16:0†source】.
6. 